# Student-selected comparison: half-life four with shorter training

**13 October 2026. The student chose both shorter-training variants.**

The confirmed public reference is the equal XGBoost/LightGBM pair with half-life four, scoring **0.95471**. The gap to 0.96000 is **0.00529**. This fixed comparison investigates training length. It does not forecast a hidden Kaggle score or automatically select a candidate.

| Fixed recipe | XGBoost trees | LightGBM trees | Recency half-life | Blend |
|---|---:|---:|---:|---|
| Current reference | 200 | 150 | 4 | 50:50 |
| Shorter LightGBM | 200 | 100 | 4 | 50:50 |
| Shorter both | 100 | 100 | 4 | 50:50 |

All other model settings remain fixed: learning rate 0.05, 15 leaves, original bin settings, original minimum-leaf settings, lambda 1, seed 42 and two CPU threads. Unknown rows retain their unknown labels and do not enter supervised fitting. Only the supplied competition files are used.

## Why this experiment

Recency weighting produced the largest confirmed recent gain without replacing the two model families. Increasing LightGBM from 150 to 225 or 300 trees previously hurt all seven later checks. Default libraries, the tested semi-supervised methods and the submitted LightGBM-only blend have not displaced the pair in the relevant comparisons. These findings motivate investigating fewer trees, but they do not establish that 100 trees is optimal or that the result will exceed 0.96.

A boosting tree makes another correction to previous predictions. Stopping earlier reduces the number of corrections at the same learning rate. The hypothesis is that the more concentrated half-life-four weights can make later corrections too specific to recent training patterns. Fewer trees can also underfit; the comparison must measure the tradeoff. Training length changes component score distributions as well, which can affect an equal probability blend.

General method reference: [LightGBM training-length documentation](https://lightgbm.readthedocs.io/en/stable/Parameters-Tuning.html#decrease-num-iterations). Its general advice is not evidence about these hidden competition labels. We retain the learning rate deliberately to isolate shorter training.

## Student modeling decision

On 13 October 2026, the student chose **“Test both shorter-training variants (Recommended)”** after reviewing the recipe and validation proposal. This executes only the fixed 200/150 reference, 200/100 variant and 100/100 variant at half-life four. No follow-on search or automatic final selection is authorised.

The settings below record that choice. The original prepared notebook is preserved.

In [1]:
from pathlib import Path
RUN_TRAINING = True
STUDENT_CHOICE = 'both'
DATA_DIR = Path('/Users/mlvlo/BT4012/Kaggle/Data Set')
OUTPUT_DIR = Path('/Users/mlvlo/.codex/.chatgpt-projects/g-p-6a901a23216481918fe5444e40decc24/student_model_review/half4_training_length_results')


In [2]:
"""Proposed fixed comparison. Default execution checks data without fitting models.

After the student's modeling choice, --run --choice both tests the two declared
shorter-training variants. --choice lgb_only tests just the LightGBM change.
There is no automatic candidate selection, follow-on search or Kaggle upload.
"""
from pathlib import Path
import argparse
import hashlib
import json
import time

import numpy as np
import pandas as pd
import sklearn
import xgboost
import lightgbm
from sklearn.metrics import roc_auc_score
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

VERSION = 'fixed-half4-training-length-v1'
FEATURES = [f'feat_{i}' for i in range(1, 166)]
HALF_LIFE = 4
FOLDS = [(k, k + 1, k + 14, 'original_14_period') for k in range(7, 22)]
FOLDS += [(k, k + 1, 35, 'later_cutoff') for k in (24, 28, 31)]
XGB_SETTINGS = dict(learning_rate=.05, tree_method='hist', grow_policy='lossguide',
                    max_depth=0, max_leaves=15, max_bin=127, min_child_weight=1,
                    reg_lambda=1, objective='binary:logistic', n_jobs=2, random_state=42)
LGB_SETTINGS = dict(learning_rate=.05, num_leaves=15, max_bin=63, min_child_samples=5,
                    min_child_weight=.001, reg_lambda=1, objective='binary',
                    deterministic=True, force_col_wise=True, verbosity=-1,
                    n_jobs=2, random_state=42)
RECIPES = {'reference_200_150': (200, 150), 'short_lgb_200_100': (200, 100),
           'short_both_100_100': (100, 100)}
EXPECTED_HASHES = {
    'Train Data.csv': '6f7e56883f0aa27147262a0d71954ad64702ca57f16fbf9ed58fae21a66dd706',
    'Test Data.csv': '344e47c8cd8815f4072d7f6ecbecee8919ab648294a8eb4ed032a026c3eb91db',
    'Sample Submission/BT4012 Competition Submission.csv':
        '3dba74ad6151053aa21f366fb9faf6da7393969a0ba6f8a2ebcec561871f739e',
}

def digest(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest()

def load_provided_data(data_dir):
    hashes = {name: digest(data_dir / name) for name in EXPECTED_HASHES}
    assert hashes == EXPECTED_HASHES, 'Provided data differs from the reviewed reference.'
    train = pd.read_csv(data_dir / 'Train Data.csv')
    test = pd.read_csv(data_dir / 'Test Data.csv')
    sample = pd.read_csv(data_dir / 'Sample Submission/BT4012 Competition Submission.csv')
    labels = pd.to_numeric(train.label, errors='coerce')
    known = train.loc[labels.notna()].copy()
    known['label'] = labels.loc[labels.notna()].astype(int)
    assert known.label.isin([0, 1]).all() and len(known) == 31235
    assert len(train) == 141772 and len(test) == len(sample) == 15329
    assert known.txId.is_unique and test['index'].is_unique
    assert sample['index'].equals(test['index'])
    assert np.isfinite(train[FEATURES].to_numpy()).all()
    assert np.isfinite(test[FEATURES].to_numpy()).all()
    return train, known, test, sample, hashes

def fitting_features(learning, cutoff):
    # Repeat the original training-only filter inside each fitting period.
    filter_rows = learning.loc[learning.time_step <= cutoff - 3]
    assert len(filter_rows) > 1 and filter_rows.time_step.max() <= cutoff - 3
    correlations = filter_rows[FEATURES].corrwith(filter_rows.time_step, method='spearman')
    removed = correlations.index[correlations.abs() > .9].tolist()
    return [c for c in FEATURES if c not in removed], removed

def component_prediction(kind, trees, learning, checking, columns, cutoff, end,
                         check_ids, hashes, output, is_final=False):
    settings = XGB_SETTINGS if kind == 'xgb' else LGB_SETTINGS
    receipt = dict(version=VERSION, data_hashes=hashes, kind=kind, trees=trees,
                   settings=settings, columns=columns, cutoff=cutoff, check_end=end,
                   half_life=HALF_LIFE, is_final=is_final,
                   versions=dict(sklearn=sklearn.__version__, xgboost=xgboost.__version__,
                                 lightgbm=lightgbm.__version__, numpy=np.__version__))
    key = hashlib.sha256(json.dumps(receipt, sort_keys=True).encode()).hexdigest()
    cache = output / 'component_predictions'
    cache.mkdir(parents=True, exist_ok=True)
    path = cache / f'{key}.npz'
    receipt_path = cache / f'{key}.json'
    if path.exists() and receipt_path.exists():
        assert json.loads(receipt_path.read_text()) == receipt
        with np.load(path, allow_pickle=False) as saved:
            assert np.array_equal(saved['checking_ids'], check_ids)
            assert np.array_equal(saved['learning_ids'], learning.txId.to_numpy())
            assert np.array_equal(saved['learning_labels'], learning.label.to_numpy())
            probabilities = saved['probabilities'].copy()
        return probabilities
    weights = (.5 ** ((learning.time_step.max() - learning.time_step) / HALF_LIFE)).to_numpy()
    weights /= weights.mean()
    model = (XGBClassifier if kind == 'xgb' else LGBMClassifier)(n_estimators=trees, **settings)
    model.fit(learning[columns], learning.label, sample_weight=weights)
    probabilities = model.predict_proba(checking[columns])[:, 1]
    assert np.isfinite(probabilities).all() and ((probabilities >= 0) & (probabilities <= 1)).all()
    if is_final:
        models = output / 'models'
        models.mkdir(exist_ok=True)
        if kind == 'xgb':
            model_path = models / f'xgb_{trees}_half4.json'
            model.save_model(model_path)
            reloaded = XGBClassifier()
            reloaded.load_model(model_path)
            reload_probabilities = reloaded.predict_proba(checking[columns])[:, 1]
        else:
            model_path = models / f'lgb_{trees}_half4.txt'
            model.booster_.save_model(str(model_path))
            reloaded = lightgbm.Booster(model_file=str(model_path))
            reload_probabilities = reloaded.predict(checking[columns])
        assert np.allclose(probabilities, reload_probabilities, atol=1e-7, rtol=0)
    temporary = path.with_suffix('.partial.npz')
    np.savez_compressed(temporary, probabilities=probabilities, checking_ids=check_ids,
                        learning_ids=learning.txId.to_numpy(), learning_labels=learning.label.to_numpy())
    temporary.replace(path)
    receipt_path.write_text(json.dumps(receipt, indent=2))
    return probabilities

def run_fixed_comparison(data_dir, output, choice='pending', run=False):
    train, known, test, sample, hashes = load_provided_data(data_dir)
    print(f'Provided rows checked: {len(known):,} confirmed, {len(train)-len(known):,} unknown; {len(test):,} test.')
    print('Reference: 200 XGBoost / 150 LightGBM trees, half-life 4; public AUC 0.95471.')
    print('Proposed variants: 200/100 and 100/100. No hidden-score forecast.')
    if not run:
        print('PREPARED ONLY: no models fitted, no prediction CSVs generated.')
        return dict(status='prepared_not_trained', rows_confirmed=len(known),
                    rows_unknown=len(train)-len(known), test_rows=len(test), choice=choice)
    if choice not in ('both', 'lgb_only'):
        raise ValueError('Record the student modeling choice before fitting this proposal.')
    chosen = RECIPES.copy()
    if choice == 'lgb_only':
        chosen.pop('short_both_100_100')
    output.mkdir(parents=True, exist_ok=True)
    declared = dict(version=VERSION, student_choice=choice, recipes=chosen, folds=FOLDS,
                    xgboost=XGB_SETTINGS, lightgbm=LGB_SETTINGS, half_life=HALF_LIFE,
                    blend_weights=[.5, .5], data_hashes=hashes, automatic_selection=False,
                    external_data=False, kaggle_upload=False,
                    feature_selection='Known fitting rows through cutoff minus three; abs Spearman > .9 removed.')
    (output / 'declared_recipe.json').write_text(json.dumps(declared, indent=2))
    started = time.monotonic()
    rows, per_step, masks = [], [], []
    for cutoff, first, last, group in FOLDS:
        learning = known.loc[known.time_step <= cutoff]
        checking = known.loc[known.time_step.between(first, last)]
        assert learning.time_step.max() < checking.time_step.min()
        xcolumns, removed = fitting_features(learning, cutoff)
        masks.append(dict(cutoff=cutoff, removed=removed, filter_fit_end=cutoff-3))
        components = {}
        for kind, counts, columns in [('xgb', {p[0] for p in chosen.values()}, xcolumns),
                                     ('lgb', {p[1] for p in chosen.values()}, FEATURES)]:
            for trees in sorted(counts):
                components[(kind, trees)] = component_prediction(
                    kind, trees, learning, checking, columns, cutoff, last,
                    checking.txId.to_numpy(), hashes, output)
        for name, (xtrees, ltrees) in chosen.items():
            p = .5*components[('xgb', xtrees)] + .5*components[('lgb', ltrees)]
            rows.append(dict(cutoff=cutoff, check_first=first, check_last=last, group=group,
                             recipe=name, training_rows=len(learning), checking_rows=len(checking),
                             auc=float(roc_auc_score(checking.label, p))))
            for step in checking.time_step.unique():
                selected = checking.time_step.to_numpy() == step
                y = checking.label.to_numpy()[selected]
                per_step.append(dict(cutoff=cutoff, step=int(step), recipe=name,
                                     rows=int(selected.sum()), positives=int(y.sum()),
                                     auc=float(roc_auc_score(y, p[selected])) if len(np.unique(y)) == 2 else None))
        pd.DataFrame(rows).to_csv(output / 'validation_results.csv', index=False)
        (output / 'progress.json').write_text(json.dumps(dict(phase='validation', last_cutoff=cutoff,
                                      completed_checks=len(masks), elapsed_seconds=time.monotonic()-started)))
        print(f'Completed training through {cutoff}, checking {first}-{last}.', flush=True)
    scores = pd.DataFrame(rows)
    summaries = []
    views = {'original_15': scores.group.eq('original_14_period'),
             'original_later_seven': scores.cutoff.between(15, 21),
             'later_cutoffs_24_28_31': scores.group.eq('later_cutoff')}
    for view, selected in views.items():
        frame = scores.loc[selected]
        control = frame.loc[frame.recipe.eq('reference_200_150')].set_index('cutoff').auc
        for name in chosen:
            values = frame.loc[frame.recipe.eq(name)].set_index('cutoff').auc
            delta = values-control
            summaries.append(dict(group=view, recipe=name, checks=len(values), mean_auc=values.mean(),
                                  worst_auc=values.min(), mean_delta_vs_reference=delta.mean(),
                                  wins_vs_reference=int((delta > 0).sum())))
    pd.DataFrame(summaries).to_csv(output / 'summary.csv', index=False)
    pd.DataFrame(per_step).to_csv(output / 'per_step_auc.csv', index=False)
    (output / 'training_only_feature_masks.json').write_text(json.dumps(masks, indent=2))
    xcolumns, removed = fitting_features(known, 35)
    assert set(removed) == {'feat_100','feat_101','feat_103','feat_136','feat_137','feat_139'}
    components = {}
    for kind, counts, columns in [('xgb', {p[0] for p in chosen.values()}, xcolumns),
                                 ('lgb', {p[1] for p in chosen.values()}, FEATURES)]:
        for trees in sorted(counts):
            components[(kind, trees)] = component_prediction(
                kind, trees, known, test, columns, 35, 49, test['index'].to_numpy(), hashes, output, True)
    submission_hashes = {}
    for name, (xtrees, ltrees) in chosen.items():
        submission = sample[['index']].copy()
        p = .5*components[('xgb', xtrees)] + .5*components[('lgb', ltrees)]
        submission['target'] = p
        assert submission['index'].equals(sample['index']) and submission.target.between(0, 1).all()
        path = output / f'submission_half4_{name}_experimental.csv'
        submission.to_csv(path, index=False)
        checked = pd.read_csv(path)
        assert checked['index'].equals(sample['index']) and np.allclose(checked.target, p, atol=1e-15, rtol=0)
        submission_hashes[name] = digest(path)
        if name == 'reference_200_150':
            old_path = data_dir.parent / 'claude_experiments/submissions/submission_lowreg_pair_recent4.csv'
            if old_path.exists():
                old = pd.read_csv(old_path)
                assert old['index'].equals(submission['index'])
                assert np.max(np.abs(old.target-p)) <= 1e-7
    assert {name: digest(data_dir/name) for name in EXPECTED_HASHES} == hashes
    (output / 'progress.json').write_text(json.dumps(dict(phase='complete', checks=len(FOLDS),
                              elapsed_seconds=time.monotonic()-started, submission_hashes=submission_hashes,
                              final_selection='student decision', kaggle_upload=False), indent=2))
    print(pd.DataFrame(summaries).to_string(index=False))
    print('Fixed comparison complete; review results before selecting a submission.')
    return summaries

def main():
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument('--data-dir', type=Path, default=Path.cwd()/'Data Set')
    parser.add_argument('--output-dir', type=Path, default=Path.cwd()/'half4_training_length_results')
    parser.add_argument('--choice', choices=['pending', 'both', 'lgb_only'], default='pending')
    parser.add_argument('--run', action='store_true')
    args = parser.parse_args()
    return run_fixed_comparison(args.data_dir, args.output_dir, args.choice, args.run)



In [3]:
result = run_fixed_comparison(DATA_DIR, OUTPUT_DIR, choice=STUDENT_CHOICE, run=RUN_TRAINING)


Provided rows checked: 31,235 confirmed, 110,537 unknown; 15,329 test.
Reference: 200 XGBoost / 150 LightGBM trees, half-life 4; public AUC 0.95471.
Proposed variants: 200/100 and 100/100. No hidden-score forecast.
Completed training through 7, checking 8-21.
Completed training through 8, checking 9-22.
Completed training through 9, checking 10-23.
Completed training through 10, checking 11-24.
Completed training through 11, checking 12-25.
Completed training through 12, checking 13-26.
Completed training through 13, checking 14-27.
Completed training through 14, checking 15-28.
Completed training through 15, checking 16-29.
Completed training through 16, checking 17-30.
Completed training through 17, checking 18-31.
Completed training through 18, checking 19-32.
Completed training through 19, checking 20-33.
Completed training through 20, checking 21-34.
Completed training through 21, checking 22-35.
Completed training through 24, checking 25-35.
Completed training through 28, checkin

## Validation and interpretation

The protocol includes the 15 existing fourteen-period checks (train through 7–21, check the next fourteen steps) and three later training cutoffs: train through 24, 28 and 31, then check through 35. The added checks use progressively shorter forecast horizons. They overlap earlier development data, including previously examined periods; they are not fresh independent holdouts.

In every check, the XGBoost time-feature filter is computed using confirmed fitting rows through cutoff minus three. Columns with absolute Spearman correlation with time above 0.9 are excluded. Reference and candidates share the same mask for each check. This removes the earlier hardcoded-mask lookahead from this comparison. Consequently, these validation AUCs should be compared with this newly fitted control rather than directly with old fixed-mask averages.

For final fitting through step 35, this procedure must reproduce the original six exclusions: feat_100, feat_101, feat_103, feat_136, feat_137 and feat_139. LightGBM retains all 165 features. The original half-life-four test predictions are checked within 1e-7 when their reference file is available.

Completed component fits have data, settings, feature-list and package-version receipts so an interrupted run can resume without repeating valid completed components. The output includes every declared candidate, per-period AUCs and group summaries. No winner is selected automatically. Later checks are reported separately from the fourteen-period checks; their different horizons should not be hidden in a single headline average.

The final files are experimental predictions pending student review. Correct CSV format, probability ranges, saved-model reloads and source hashes verify reproducibility; they do not guarantee prediction quality. Kaggle uploading and final-assessment selection remain student decisions.

## AI assistance record

OpenAI Codex reviewed prior results, proposed and explained this fixed comparison and prepared its implementation. The student explicitly selected both shorter-training variants. Codex executes that recipe and verifies its outputs. The student remains responsible for interpreting results and choosing any Kaggle submission. No automatic tuning or upload is implemented.

### 42.1. Student-selected experiment and measured result

**13 October 2026.** After reviewing the proposed recipe, the student chose **“Test both shorter-training variants (Recommended)”**. Codex fitted the half-life-four reference (200 XGBoost trees / 150 LightGBM trees) and the two declared alternatives (200/100 and 100/100). Half-life four, the 50:50 probability average, learning rate 0.05, leaf limits, bin settings, minimum-leaf constraints, lambda 1, seed 42 and two CPU threads remained fixed.

The run completed **18 chronological checks**, **76 fresh component fits**, and full fitting on all **31,235 confirmed labels** in **52.3 seconds**. The **110,537 unknown training rows** received no invented labels. Each recipe produced **15,329 aligned experimental test predictions**. There was no follow-on parameter search or Kaggle upload.

| Check group | Reference 200/150 | Shorter LightGBM 200/100 | Shorter both 100/100 |
|---|---:|---:|---:|
| Original 15 | 0.950128 | 0.948218 | 0.946180 |
| Original later seven | 0.915564 | 0.912546 | 0.908188 |
| Three later cutoffs | 0.997337 | 0.997151 | 0.996751 |

Relative to the reference, shorter LightGBM changes the original-15 mean by **-0.001910**, winning **3/15** checks. Shorter both changes it by **-0.003947**, winning **2/15**. On the original later-seven checks they win **2/7** and **0/7**, respectively. Both lose **all three** added later-cutoff checks. Their worst original-check scores also fall relative to the reference.

**Assessment:** Neither shorter-training variant supports replacing the 0.95471 public reference. Both have lower original-15 and later-seven means and both lose all three later-cutoff checks. The result does not support spending a submission on either new CSV. The fixed current recipe remains the reference; no new settings are selected or fitted automatically.

The proposed hypothesis was that fewer boosting corrections might reduce overfitting under concentrated recency weights. These results do not support that hypothesis for the chosen reductions. Underfitting is a plausible explanation, but lower checking scores alone do not establish a unique cause or show that counts above the reference will improve performance.

### Later training endpoints and their limits

| Train through | Check steps | Reference 200/150 | Shorter LightGBM 200/100 | Shorter both 100/100 |
|---:|---|---:|---:|---:|
| 21 | 22–35 | 0.903163 | 0.895976 | 0.899291 |
| 24 | 25–35 | 0.997268 | 0.997136 | 0.996865 |
| 28 | 29–35 | 0.996713 | 0.996289 | 0.995625 |
| 31 | 32–35 | 0.998030 | 0.998028 | 0.997763 |

The three added checks forecast **11, 7 and 4** periods, respectively. Their AUCs are close to 0.997 for every tested recipe. This is a result for those supplied training/checking periods, not an expected Kaggle AUC. It cannot be equated with the fourteen-period test (steps 36–49). Their shorter horizons, different checking periods, overlap and previously examined labels all remain important limits. Every check has disjoint fitting/checking transaction IDs, and no checking row fits a model or filter.

The original 15 checks also overlap. Across all 18 checks, **181,071 checking appearances** cover **21,682 unique transactions**. The reported means and win counts are development diagnostics, not independent repetitions or calibrated confidence intervals.

### Feature filtering and comparison fairness

For every check, XGBoost's time-feature filter uses only confirmed fitting rows through cutoff minus three, removing features whose absolute Spearman correlation with time exceeds 0.9. Reference and variants share that cutoff-specific mask. This repairs the hardcoded-mask lookahead in the older recency development checks. Therefore, compare these candidate AUCs with this newly fitted reference, rather than treating the small change in the reference's old all-15 mean as a model improvement or decline.

Final fitting through step 35 produces the same six exclusions as the public reference: feat_100, feat_101, feat_103, feat_136, feat_137 and feat_139. LightGBM uses all 165 features. The regenerated reference test probabilities match the stored 0.95471 file within **5.9604645e-08**. The exact previously submitted reference is preserved.

### Verification, files and next decision

An independent verification process recalculated every AUC, per-period score, mean, worst score, delta and win count from the saved component probabilities. It checked fitting/checking labels and identifiers, nested feature masks, all fixed settings, source hashes and candidate probability formulas. All four final models were reloaded in a fresh process and reproduced their saved probabilities. Actual final tree counts match 100/200 for XGBoost and 100/150 for LightGBM. Every CSV passes column, row-count, exact sample/test order, finite-score, probability-range and round-trip checks. No extra model fitting was needed for verification.

The two new prediction files are preserved for reproducibility, although they are not recommended submissions on the available evidence:

- submission_half4_short_lgb_200_100_experimental.csv
- submission_half4_short_both_100_100_experimental.csv

Their public scores remain unknown. No estimate above 0.96 follows from this run. The verified public best remains **0.95471**, **0.00529 below** 0.96000. No final-assessment selection was made by Codex. A new material experiment requires a student choice; this selected comparison is complete and will not be repeated automatically.

**AI assistance record:** OpenAI Codex reviewed prior work, proposed and explained this fixed comparison, and prepared the implementation. The student selected both variants and the stated checking protocol. Codex executed only that selected comparison, verified its artifacts and drafted this factual journal record. The student remains responsible for interpretation, subsequent modeling decisions, submission selection and disclosure. No external transaction data or labels, guessed unknown labels, AutoML, automatic parameter search or Kaggle upload was used.

The standalone executed notebook is e1156285_half4_training_length.ipynb. The exact source, declared settings, component predictions, four full models, all checking scores, three final CSVs and independent verification are in half4_training_length_results/. To reproduce on another machine, adjust the notebook's DATA_DIR and OUTPUT_DIR to the supplied data and desired output folders; no prior experiment is required for fitting.


**Component diagnostic:** Existing saved component predictions also show lower group means for LightGBM 100 versus 150 and XGBoost 100 versus 200 in all three reported groups. Thus the negative blend result is accompanied by weaker individual ranking, rather than only a change in how probability scales combine. These are diagnostics of the already-fitted components; no additional candidate, blend weight or model fit was introduced. Full values are in component_auc_diagnostic_summary.csv.
